# 02 — Vessel Segmentation Preprocessing
**Dataset:** DRIVE — Digital Retinal Images for Vessel Extraction (`andrewmvd/drive-digital-retinal-images-for-vessel-extraction`)
**Goal:** Download via KaggleHub, preprocess fundus images + vessel masks, save as `.npz`, visualize samples.
> Run on **Google Colab**. Set your Kaggle credentials before running.

## 1. Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'kagglehub', 'opencv-python-headless', 'tqdm', 'matplotlib', 'scikit-learn', 'numpy', 'pandas'],
               check=True)
print('Dependencies ready.')

## 2. Kaggle Authentication

In [ ]:
import os
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists() and 'google.colab' in sys.modules:
    from google.colab import files
    print('Upload your kaggle.json file:')
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        kaggle_json.write_bytes(uploaded['kaggle.json'])
        kaggle_json.chmod(0o600)

# os.environ['KAGGLE_USERNAME'] = 'your_username'
# os.environ['KAGGLE_KEY'] = 'your_api_key'

if kaggle_json.exists():
    kaggle_json.chmod(0o600)
    print('Kaggle credentials ready.')
else:
    print('WARNING: kaggle.json not found.')

## 3. Repository Setup & Imports

In [ ]:
import sys
import shutil
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from engine.image_preprocessing import PreprocessConfig, preprocess_fundus_image, save_preprocessed

data_dir = repo_root / 'data'
raw_dir = data_dir / 'raw'
processed_dir = data_dir / 'processed'
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print('Repo root:', repo_root)

## 4. Download DRIVE Dataset via KaggleHub

In [ ]:
import kagglehub

DATASET_SLUG = 'andrewmvd/drive-digital-retinal-images-for-vessel-extraction'
drive_root = raw_dir / 'drive'
drive_root.mkdir(parents=True, exist_ok=True)

already_downloaded = any(drive_root.rglob('*.png')) or any(drive_root.rglob('*.jpg')) or any(drive_root.rglob('*.tif'))
if not already_downloaded:
    print(f'Downloading {DATASET_SLUG} ...')
    download_path = Path(kagglehub.dataset_download(DATASET_SLUG))
    if download_path != drive_root:
        shutil.copytree(download_path, drive_root, dirs_exist_ok=True)
    print('Download complete:', drive_root)
else:
    print('Dataset already present at:', drive_root)

print(f'Total files: {len(list(drive_root.rglob("*")))}' )

## 5. Index Images & Vessel Masks

In [ ]:
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}
MASK_KEYWORDS = ['mask', 'manual', '1st', '2nd', 'vessel', 'groundtruth', 'gt']

def is_vessel_mask(path):
    tokens = path.name.lower() + ' ' + ' '.join(p.name.lower() for p in path.parents)
    return any(k in tokens for k in MASK_KEYWORDS)

all_imgs = [p for p in drive_root.rglob('*') if p.suffix.lower() in IMAGE_EXTS and not is_vessel_mask(p)]
all_masks = [p for p in drive_root.rglob('*') if p.suffix.lower() in IMAGE_EXTS and is_vessel_mask(p)]

# Match by stem
mask_map = {m.stem: m for m in all_masks}
rows = [{'image_path': img, 'mask_path': mask_map.get(img.stem)} for img in sorted(all_imgs)]
df = pd.DataFrame(rows)
print(f'Images: {len(df)} | With masks: {df.mask_path.notna().sum()}')
df.head()

## 6. Train / Val / Test Split (70/15/15)

In [ ]:
if df.empty:
    raise RuntimeError('No images found — check dataset download path.')

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
splits = {'train': train_df, 'val': val_df, 'test': test_df}

## 7. Batch Preprocessing & Save

In [ ]:
config = PreprocessConfig(target_size=(512, 512), normalization='zero_one')
DATASET_NAME = 'drive'
errors = []

for split_name, split_df in splits.items():
    out_dir = processed_dir / DATASET_NAME / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'\nProcessing {split_name} ({len(split_df)} images)...')
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=split_name):
        try:
            mask = row['mask_path'] if pd.notna(row.get('mask_path')) else None
            result = preprocess_fundus_image(row['image_path'], mask=mask, config=config)
            out_file = out_dir / (row['image_path'].stem + '.npz')
            save_preprocessed(result, out_file)
        except Exception as e:
            errors.append({'file': str(row['image_path']), 'error': str(e)})
            print(f'  WARNING: skipped {row["image_path"].name} — {e}')

print(f'\nDone. Errors: {len(errors)}')

## 8. Dataset Statistics

In [ ]:
print('=== Dataset Statistics ===')
for split_name, split_df in splits.items():
    print(f'{split_name}: {len(split_df)} images')

widths, heights = [], []
for _, row in df.iterrows():
    img = cv2.imread(str(row['image_path']))
    if img is not None:
        h, w = img.shape[:2]
        widths.append(w); heights.append(h)

if widths:
    print(f'\nImage sizes (W x H):')
    print(f'  Width  — min:{min(widths)} max:{max(widths)} mean:{int(np.mean(widths))}')
    print(f'  Height — min:{min(heights)} max:{max(heights)} mean:{int(np.mean(heights))}')

## 9. Visualization — Sample Grid (Image + Vessel Mask)

In [ ]:
samples = train_df[train_df['mask_path'].notna()].sample(min(4, len(train_df)), random_state=42)

fig, axes = plt.subplots(len(samples), 2, figsize=(10, 4 * len(samples)))
if len(samples) == 1:
    axes = [axes]

for i, (_, row) in enumerate(samples.iterrows()):
    img_bgr = cv2.imread(str(row['image_path']))
    if img_bgr is None: continue
    axes[i][0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    axes[i][0].set_title(f'Image: {row["image_path"].name}', fontsize=9)
    axes[i][0].axis('off')
    if pd.notna(row.get('mask_path')):
        m = cv2.imread(str(row['mask_path']), cv2.IMREAD_GRAYSCALE)
        if m is not None:
            axes[i][1].imshow(m, cmap='gray')
            axes[i][1].set_title('Vessel Mask', fontsize=9)
    axes[i][1].axis('off')

plt.suptitle('DRIVE — Sample Images & Vessel Masks', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Next Steps
- Feed `.npz` files from `data/processed/drive/` into a vessel segmentation model
- Recommended architecture: **U-Net** with skip connections or **Attention U-Net**
- Metrics: **Dice**, **IoU**, **Sensitivity**, **Specificity**, **AUC-ROC**
- DRIVE is small (40 images) — use strong augmentation: flips, rotations, elastic deformation